## HIL-001 Data Inspection and Cleaning

This notebook documents a conservative cleaning workflow for the HIL-001 semiconductor businesses snapshot. The raw ArcGIS response is preserved unchanged; derived cleaned attributes and quality flags are exported separately.

- Source: City of Hillsboro GIS
- Dataset: Semiconductor businesses
- Snapshot: `2026-08-26`
- Geometry: Point
- Coordinate system: NAD 1983 HARN StatePlane Oregon North FIPS 3601


In [9]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_VERSION = "2026-08-26"
DATA_ROOT = PROJECT_ROOT / DATA_VERSION
RAW_DIR = DATA_ROOT / "raw"
PROCESSED_DIR = DATA_ROOT / "processed"
RAW_FILE = RAW_DIR / "HIL-001.json"
PROCESSED_FILE = PROCESSED_DIR / "HIL-001_cleaned.json"
MANIFEST_FILE = PROCESSED_DIR / "HIL-001_cleaning_manifest.json"

with open(RAW_FILE, "r", encoding="utf-8") as file:
    raw_data = json.load(file)

df = pd.DataFrame([feature["attributes"] for feature in raw_data["features"]])
geometry_df = pd.DataFrame([feature.get("geometry", {}) for feature in raw_data["features"]])

print(f"Loaded {RAW_FILE.name}: {len(df):,} rows, {len(df.columns):,} source fields")
print(f"Geometry type: {raw_data.get('geometryType')}")

Loaded HIL-001.json: 59 rows, 25 source fields
Geometry type: esriGeometryPoint


## Raw Structure and Missingness

The first inspection separates missing values from valid zero values and checks the source identifiers before any transformation.

In [10]:
schema_df = pd.DataFrame(raw_data["fields"])[["name", "alias", "type"]]
missing = df.isna().sum().to_frame("missing_count")
missing["missing_percent"] = (missing["missing_count"] / len(df) * 100).round(1)

print("Top-level keys:", list(raw_data))
print("Field types:")
display(schema_df)
print("Missingness:")
display(missing.sort_values("missing_count", ascending=False))


Top-level keys: ['objectIdFieldName', 'globalIdFieldName', 'geometryType', 'spatialReference', 'fields', 'features']
Field types:


,name,alias,type
0,OBJECTID,OBJECTID,esriFieldTypeOID
1,OWNER,Owner,esriFieldTypeString
2,PRODUCT,Product,esriFieldTypeString
3,LOGO,Logo,esriFieldTypeString
4,INDUSTRY,Industry,esriFieldTypeString
5,NAICS_CODE,NAICS Code,esriFieldTypeInteger
6,BUILDING_SF,Building Square Footage,esriFieldTypeInteger
7,SUB_INDUSTRY,Sub-Industry,esriFieldTypeString
8,FAC_TYPE_BACK_OFFICE,Facility Type Back Office,esriFieldTypeString
9,FAC_TYPE_DISTRIBUTION,Facility Type Distribution,esriFieldTypeString


Missingness:


,missing_count,missing_percent
NOTES,59,100.0
YEARS_IN_HILLSBORO,58,98.3
BUILDING_SF,51,86.4
SUPPLY_CHAIN,29,49.2
FAC_TYPE_HQ,27,45.8
FAC_TYPE_BACK_OFFICE,26,44.1
FAC_TYPE_MANUFACTURING,26,44.1
FAC_TYPE_RD,26,44.1
FAC_TYPE_OFFICE,26,44.1
FAC_TYPE_SALES_SERVICE,26,44.1


## Quality Findings

- `NOTES` is empty for every record and is excluded from the compact analytical view only.
- `YEARS_IN_HILLSBORO` and `BUILDING_SF` are largely missing; no values are imputed.
- `OBJECTID` and `GlobalID` are unique in this snapshot.
- Facility-type fields use `Yes`/`No`; unexpected values are flagged rather than guessed.
- Point coordinates are retained in the source State Plane coordinate system and checked for finite values.

In [11]:
identifier_audit = pd.DataFrame({
    "field": ["OBJECTID", "GlobalID"],
    "unique_values": [df["OBJECTID"].nunique(), df["GlobalID"].nunique()],
    "missing_values": [df["OBJECTID"].isna().sum(), df["GlobalID"].isna().sum()],
    "duplicate_values": [df["OBJECTID"].duplicated().sum(), df["GlobalID"].duplicated().sum()],
})

facility_columns = [column for column in df.columns if column.startswith("FAC_TYPE_")]
facility_values = sorted(pd.unique(df[facility_columns].stack().dropna()))
coordinate_audit = pd.DataFrame({
    "field": ["x", "y"],
    "missing": [geometry_df["x"].isna().sum(), geometry_df["y"].isna().sum()],
    "finite": [np.isfinite(geometry_df["x"]).sum(), np.isfinite(geometry_df["y"]).sum()],
})

print("Identifier audit")
display(identifier_audit)
print("Facility values:", facility_values)
print("Coordinate audit")
display(coordinate_audit)


Identifier audit


,field,unique_values,missing_values,duplicate_values
0,OBJECTID,59,0,0
1,GlobalID,59,0,0


Facility values: ['No', 'Yes']
Coordinate audit


,field,missing,finite
0,x,0,59
1,y,0,59


## Derived Clean Analytical View

Cleaning is limited to representation and quality flags: whitespace-only strings become missing, ArcGIS epoch dates become ISO-8601 strings, and the raw attributes and geometry remain available in the source snapshot.

In [12]:
analysis_df = df.copy()

# Normalize blank text without changing meaningful source values.
text_columns = analysis_df.select_dtypes(include=["str"]).columns
for column in text_columns:
    analysis_df[column] = analysis_df[column].map(
        lambda value: None if isinstance(value, str) and not value.strip() else value
    )

# Convert ArcGIS epoch milliseconds to explicit ISO-8601 values.
for column in ["UTC_CreateDate", "UTC_EditDate"]:
    analysis_df[column] = pd.to_datetime(
        analysis_df[column], unit="ms", utc=True, errors="coerce"
    ).dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    analysis_df[column] = analysis_df[column].where(analysis_df[column].notna(), None)

valid_facility_values = {"Yes", "No"}
analysis_df["FACILITY_VALUE_QA_FLAG"] = analysis_df[facility_columns].apply(
    lambda row: any(value not in valid_facility_values for value in row.dropna()), axis=1
)
analysis_df["COORDINATE_QA_FLAG"] = ~(
    geometry_df["x"].map(np.isfinite) & geometry_df["y"].map(np.isfinite)
)
analysis_df["EMPLOYEE_NOTE_QA_FLAG"] = (
    analysis_df["NUMBER_EMP_NOTES"].notna()
    & analysis_df["NUMBER_EMP"].isna()
)
analysis_df["OBJECTID_DUPLICATE_QA_FLAG"] = analysis_df["OBJECTID"].duplicated(keep=False)
analysis_df["GLOBALID_DUPLICATE_QA_FLAG"] = analysis_df["GlobalID"].duplicated(keep=False)

completely_missing_fields = [
    column for column in df.columns if df[column].isna().all()
]
analysis_df_compact = analysis_df.drop(columns=completely_missing_fields)

print("Completely missing source fields:", completely_missing_fields)
print("Derived analytical fields:", len(analysis_df_compact.columns))
print("QA flag totals:")
display(analysis_df.filter(like="_QA_FLAG").sum().to_frame("flagged_records"))


Completely missing source fields: ['NOTES']
Derived analytical fields: 29
QA flag totals:


,flagged_records
FACILITY_VALUE_QA_FLAG,0
COORDINATE_QA_FLAG,0
EMPLOYEE_NOTE_QA_FLAG,2
OBJECTID_DUPLICATE_QA_FLAG,0
GLOBALID_DUPLICATE_QA_FLAG,0


## Validation and Export

Rows and geometries are preserved one-for-one. The processed file contains compact attributes, derived QA fields, and the original point geometry; the raw file is never overwritten.

In [13]:
assert len(raw_data["features"]) == len(analysis_df) == 59
assert analysis_df["OBJECTID"].is_unique
assert analysis_df["GlobalID"].is_unique
assert not analysis_df["COORDINATE_QA_FLAG"].any()
assert set(pd.unique(df[facility_columns].stack().dropna())) <= valid_facility_values
assert analysis_df_compact.columns.is_unique

print("Cleaning validation passed")
print(f"Rows preserved: {len(analysis_df):,}")
print(f"Compact fields: {len(analysis_df_compact.columns):,}")
print(f"Employee-note review flags: {int(analysis_df['EMPLOYEE_NOTE_QA_FLAG'].sum())}")


Cleaning validation passed
Rows preserved: 59
Compact fields: 29
Employee-note review flags: 2


In [14]:
import math
from datetime import datetime


def json_safe(value):
    if value is None or value is pd.NA:
        return None
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and math.isnan(value):
        return None
    return value

schema_records = [field.copy() for field in raw_data["fields"]]
for field_name in analysis_df_compact.columns:
    if field_name not in {field["name"] for field in schema_records}:
        schema_records.append({
            "name": field_name,
            "alias": field_name,
            "type": "esriFieldTypeInteger",
        })

processed_features = []
for index, raw_feature in enumerate(raw_data["features"]):
    attributes = {
        field: json_safe(value)
        for field, value in analysis_df_compact.iloc[index].to_dict().items()
    }
    processed_features.append({
        "attributes": attributes,
        "geometry": raw_feature.get("geometry"),
    })

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
processed_data = {
    "objectIdFieldName": raw_data.get("objectIdFieldName"),
    "globalIdFieldName": raw_data.get("globalIdFieldName"),
    "geometryType": raw_data.get("geometryType"),
    "spatialReference": raw_data.get("spatialReference"),
    "fields": schema_records,
    "features": processed_features,
    "cleaning_summary": {
        "source_file": RAW_FILE.name,
        "data_version": DATA_VERSION,
        "rows": len(processed_features),
        "compact_attribute_fields": len(analysis_df_compact.columns),
        "completely_missing_source_fields": completely_missing_fields,
        "employee_note_review_flags": int(analysis_df["EMPLOYEE_NOTE_QA_FLAG"].sum()),
    },
}

with open(PROCESSED_FILE, "w", encoding="utf-8") as file:
    json.dump(processed_data, file, indent=2, ensure_ascii=True)

In [15]:
cleaning_manifest = [
    {
        "field_or_scope": "Blank text values",
        "action": "Represent blank or whitespace-only strings as missing in derived attributes",
        "source_preserved": True,
        "review_flag": "None",
    },
    {
        "field_or_scope": "UTC_CreateDate and UTC_EditDate",
        "action": "Convert ArcGIS epoch milliseconds to ISO-8601 UTC strings",
        "source_preserved": True,
        "review_flag": "None",
    },
    {
        "field_or_scope": "NOTES",
        "action": "Exclude completely missing field from compact analytical view",
        "source_preserved": True,
        "review_flag": "None",
    },
    {
        "field_or_scope": "Facility flags and point coordinates",
        "action": "Retain source values and add validation flags",
        "source_preserved": True,
        "review_flag": "FACILITY_VALUE_QA_FLAG / COORDINATE_QA_FLAG",
    },
    {
        "field_or_scope": "NUMBER_EMP_NOTES without NUMBER_EMP",
        "action": "Retain values and flag for review",
        "source_preserved": True,
        "review_flag": "EMPLOYEE_NOTE_QA_FLAG",
    },
]

manifest_data = {
    "dataset": "HIL-001",
    "dataset_name": "semiconductor_businesses",
    "data_version": DATA_VERSION,
    "created_utc": datetime.now().astimezone().isoformat(),
    "source_file": str(RAW_FILE),
    "processed_file": str(PROCESSED_FILE),
    "rows": len(processed_features),
    "cleaning_manifest": cleaning_manifest,
}

with open(MANIFEST_FILE, "w", encoding="utf-8") as file:
    json.dump(manifest_data, file, indent=2, ensure_ascii=True)

with open(PROCESSED_FILE, "r", encoding="utf-8") as file:
    reloaded_processed_data = json.load(file)
with open(MANIFEST_FILE, "r", encoding="utf-8") as file:
    reloaded_manifest_data = json.load(file)

assert len(reloaded_processed_data["features"]) == len(raw_data["features"])
assert len(reloaded_manifest_data["cleaning_manifest"]) == len(cleaning_manifest)
assert reloaded_processed_data["features"][0]["attributes"]["OBJECTID"] == int(df.iloc[0]["OBJECTID"])
print(f"Wrote: {PROCESSED_FILE}")
print(f"Wrote: {MANIFEST_FILE}")
print("Reload validation passed")


Wrote: c:\Users\John\Documents\hillsborogis\2026-08-26\processed\HIL-001_cleaned.json
Wrote: c:\Users\John\Documents\hillsborogis\2026-08-26\processed\HIL-001_cleaning_manifest.json
Reload validation passed
